# 07 — Formal H3 Adjudication (Target-Support Redesign)

**Scope.** This is the FINAL adjudication of the already-completed H3
`A_TARGET_SUPPORT_REDESIGN` intervention against the prospectively frozen H3
criteria. It is a **disk-only** adjudication: no training, no inference, no new
experiment, no TEST access, no threshold/checkpoint/gate change.

**Predecessor state:** `H3_DOWNSTREAM_EVALUATION_COMPLETE__FORMAL_ADJUDICATION_PENDING`

**This notebook produces the frozen scientific verdict — it does not create one.**
The verdict is read from the prospectively frozen criteria artifact and applied to
the authoritative observed values.

**Governance.** TEST and external TEST remain CLOSED. Phase 15 remains
`NOT_STARTED_AND_NOT_AUTHORIZED`. Options B and C are future-only and are NOT
started here.

## 1. Scope and governance

In [1]:
import json, hashlib, sys
from fractions import Fraction
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path('D:/Softfors/jup')
AUDITS = ROOT / 'reports' / 'audits'
TABLES = ROOT / 'reports' / 'tables'
FIGDIR = ROOT / 'reports' / 'figures' / 'h3_adjudication'
FIGDIR.mkdir(parents=True, exist_ok=True)

def load(p): return json.loads((ROOT / p).read_text(encoding='utf-8'))
def sha(p):
    h = hashlib.sha256()
    with open(ROOT / p, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''): h.update(c)
    return h.hexdigest()

ADJ = load('reports/audits/h3_formal_adjudication.json')
print('adjudication artifact :', ADJ['artifact'])
print('status                :', ADJ['status'])
print('scientific label      :', ADJ['scientific_label'])
print('adjudication sha256   :', sha('reports/audits/h3_formal_adjudication.json'))
print()
print('This notebook performs no training, no inference and no TEST access.')

adjudication artifact : h3_formal_adjudication
status                : H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPORT_INSUFFICIENT__PHASE15_LOCKED
scientific label      : H3_SUPPLY_REDESIGN_INSUFFICIENT
adjudication sha256   : f07ff0c32499b3b05c42f176b21b2cd25329fc06fade7ab90f0f222586e71fb2

This notebook performs no training, no inference and no TEST access.


## 2. H3 scientific question

In [2]:
q = ADJ['scientific_question']
for k in ['axis', 'intervention_family', 'mechanistic_hypothesis', 'frozen_mapping']:
    print(f'{k:26s}: {q[k]}')
print(f"{'A_ref (authoritative)':26s}: {q['A_ref_authoritative']}")
print()
print(q['A_ref_note'])

axis                      : A_PROPOSAL_SUPPLY_RECALL_LIMITATION
intervention_family       : A_TARGET_SUPPORT_REDESIGN
mechanistic_hypothesis    : FIXED_TARGET_SUPPORT_MAY_BE_MISMATCHED_TO_COMPONENT_SCALE
frozen_mapping            : sigma_i = clip(0.35 + 0.40 * sqrt(A128_i / A_ref), 0.35, 0.75)
A_ref (authoritative)     : 13.0

A_ref = 13.0 is the AUTHORITATIVE executing value: it is bound in the final pretraining binding v2, the training launch binding, and the frozen H3 config (configs/kub/phase13b_localizer_h3_size_adaptive_sigma.yaml). The earlier design-correction artifact froze A_ref = 14.0 (p75 of the code-exact 128-space TRAIN area distribution); that value was SUPERSEDED before training by the v2 calibration (770 images / 1161 components) and is retained only as provenance.


## 3. Frozen intervention

In [3]:
proto = load('reports/audits/h3_proposal_supply_design_protocol.json')
print('design status       :', proto['design_status'])
print('selected axis       :', proto['selected_axis'])
print('priority rule fired :', proto['axis_priority_rule_fired'])
print('intervention        :', proto['intervention']['selected_option'])
print()
print(proto['intervention']['definition'])

design status       : H3_DESIGN_FROZEN__READY_FOR_PROSPECTIVE_TRAINING_AUTHORIZATION
selected axis       : A_PROPOSAL_SUPPLY_RECALL_LIMITATION
priority rule fired : A_PRIORITY
intervention        : A_TARGET_SUPPORT_REDESIGN

Make the localizer training target size-adaptive: replace the single fixed sigma=0.75 in make_gaussian_heatmap with a per-component sigma derived from the component's area in heatmap space (smaller sigma for tiny components -> sharper, higher-contrast peak; capped at the historical 0.75 for large components). Canonical center pixel stays 1.0; pixelwise-max fusion unchanged.


## 4. H3 evidence chain

### Table A — H3 evidence timeline

In [4]:
chain = [
    ('01 root cause', 'reports/audits/phase14_diag_root_cause.json', 'COMPLETE', 'F_MIXED_BOTTLENECK'),
    ('02 axis priority', 'reports/audits/post_phase14_h3_axis_priority_results.json', 'COMPLETE', 'A_PRIORITY (R_A=1.0, R_C=0.0)'),
    ('03 design/binding', 'reports/audits/h3_proposal_supply_final_pretraining_binding_v2.json', 'COMPLETE', 'H3_FINAL_PRETRAINING_BINDING_COMPLETE'),
    ('04 training', 'reports/audits/h3_proposal_supply_training_closure.json', 'COMPLETE', 'best epoch 54; replay PASS'),
    ('05 supply', 'reports/audits/h3_supply_axis_evaluation.json', 'COMPLETE', 'zero-prop GT 79->27; TINY 57->22'),
    ('05b reconciliation', 'reports/audits/h3_supply_axis_metric_domain_reconciliation.json', 'RECONCILED', 'proxy -0.60 INVALID for formal use'),
    ('06 downstream', 'reports/audits/h3_downstream_evaluation.json', 'COMPLETE', 'TinyRecall_H3 59/72; R_H3 2/15'),
    ('06b correction', 'reports/audits/h3_downstream_metric_domain_correction.json', 'CORRECTED', 'detection domain formal'),
    ('07 adjudication', 'reports/audits/h3_formal_adjudication.json', 'COMPLETE', 'H3_SUPPLY_REDESIGN_INSUFFICIENT'),
]
rows = []
for step, path, state, key in chain:
    rows.append({'step': step, 'artifact': Path(path).name, 'state': state,
                 'key result': key, 'sha256': (sha(path)[:16] + '...') if (ROOT / path).exists() else 'MISSING'})
display(pd.DataFrame(rows))

,step,artifact,state,key result,sha256
0,01 root cause,phase14_diag_root_cause.json,COMPLETE,F_MIXED_BOTTLENECK,dd9b5b86ab1933e7...
1,02 axis priority,post_phase14_h3_axis_priority_results.json,COMPLETE,"A_PRIORITY (R_A=1.0, R_C=0.0)",bebc56970a300fa9...
2,03 design/binding,h3_proposal_supply_final_pretraining_binding_v...,COMPLETE,H3_FINAL_PRETRAINING_BINDING_COMPLETE,ff10d13bd151d48f...
3,04 training,h3_proposal_supply_training_closure.json,COMPLETE,best epoch 54; replay PASS,d96d334d0e4d2554...
4,05 supply,h3_supply_axis_evaluation.json,COMPLETE,zero-prop GT 79->27; TINY 57->22,00e6f41351d72581...
5,05b reconciliation,h3_supply_axis_metric_domain_reconciliation.json,RECONCILED,proxy -0.60 INVALID for formal use,457e87a7b51b378d...
6,06 downstream,h3_downstream_evaluation.json,COMPLETE,TinyRecall_H3 59/72; R_H3 2/15,9bfbf32f4ddaee7b...
7,06b correction,h3_downstream_metric_domain_correction.json,CORRECTED,detection domain formal,ba85a57861263a6c...
8,07 adjudication,h3_formal_adjudication.json,COMPLETE,H3_SUPPLY_REDESIGN_INSUFFICIENT,f07ff0c32499b3b0...


## 5. Selected checkpoint binding

In [5]:
sc = ADJ['selected_checkpoint']
print('experiment id      :', ADJ['experiment_id'])
print('path               :', sc['path'])
print('sha256             :', sc['sha256'])
print('expected sha256    :', sc['sha256_expected'])
print('sha matches        :', sc['sha256_matches_expected'])
print('selected epoch     :', sc['selected_epoch'])
print('heatmap resolution :', sc['heatmap_resolution'])
print('replay verdict     :', ADJ['checkpoint_selection_replay_verdict'])
print('closure status     :', ADJ['training_closure_status'])

experiment id      : h3_kub_localizer_sigma_adaptive_sizeaware_seed42_clean_v3
path               : checkpoints/h3_kub_localizer_sigma_adaptive_sizeaware_seed42_clean_v3/best.pt
sha256             : d875cac7d09bbc3d32232191f0af2c138e5a74682fc63e4d6e01f30345eca75e
expected sha256    : d875cac7d09bbc3d32232191f0af2c138e5a74682fc63e4d6e01f30345eca75e
sha matches        : True
selected epoch     : 54
heatmap resolution : 128
replay verdict     : H3_CHECKPOINT_SELECTION_REPLAY_PASS
closure status     : H3_TRAINING_COMPLETE__CHECKPOINT_SELECTION_REPLAY_PASS__SUPPLY_EVALUATION_PENDING


## 6. Trainer multi-SHA provenance

In [6]:
tl = ADJ['provenance_caveats']['trainer_lineage']
for k in ['design_era_trainer', 'epochs_1_2_trainer', 'epochs_3_69_trainer']:
    print(f'{k:20s}: {tl[k]}')
print()
print(tl['note'])
print('single-SHA claim   :', tl['single_sha_claim'])
print('classification     :', tl['classification'])

design_era_trainer  : 37b64310d38222310ce060803e37d17e4b6916ce7c522d925234167dc26da366
epochs_1_2_trainer  : 9099d6de339aba43d5e92284ea741568f16c7f5c39e7c189c595301ca5cb88f9
epochs_3_69_trainer : 45bef74f4eddb2e376fd768b1da2e0cba743897e30a725241ac6882d60ad3578

The exact source bytes for the epoch-1/2 trainer are unavailable locally. The selected epoch-54 checkpoint was produced under the final segment trainer (45bef74f).
single-SHA claim   : False
classification     : PROVENANCE_NOT_A_POST_HOC_SCIENTIFIC_GATE


## 7. Frozen success criteria

### Table B — Frozen criteria

In [7]:
fc = ADJ['frozen_criteria']; cr = ADJ['criterion_results']
tb = pd.DataFrame([
    {'criterion': 'A', 'frozen threshold': fc['criterion_A']['rule'],
     'observed': cr['criterion_A']['observed'], 'result': cr['criterion_A']['result'],
     'role': 'NECESSARY (supply)'},
    {'criterion': 'B', 'frozen threshold': fc['criterion_B']['rule'],
     'observed': cr['criterion_B']['observed'], 'result': cr['criterion_B']['result'],
     'role': 'NECESSARY (supply)'},
    {'criterion': 'C', 'frozen threshold': fc['criterion_C']['rule'],
     'observed': f"{ADJ['observed']['formal_R_H3']:.10f}", 'result': cr['criterion_C']['result'],
     'role': 'ADJUDICATION GATE (downstream)'},
])
display(tb)
print('contamination policy:', fc['contamination'])

,criterion,frozen threshold,observed,result,role
0,A,zero_proposal_TINY <= 20/72,22/72,FAIL,NECESSARY (supply)
1,B,zero_proposal_GT <= 35/152,27/152,PASS,NECESSARY (supply)
2,C,formal downstream R_H3 >= 0.30,0.1333333333,FAIL,ADJUDICATION GATE (downstream)


contamination policy: OBSERVATIONAL_ONLY__NOT_GATED


## 8. Frozen failure boundary

### Table C — Failure boundary

In [8]:
fb = ADJ['failure_boundary']
tc = pd.DataFrame([
    {'clause': '1', 'rule': fb['clause_1']['rule'], 'observed': fb['clause_1']['observed'],
     'fired': fb['clause_1']['fired']},
    {'clause': '2', 'rule': fb['clause_2']['rule'], 'observed': f"{fb['clause_2']['observed']:.10f}",
     'fired': fb['clause_2']['fired']},
])
display(tc)
print('overall failure fired:', fb['fired'])
print('intermediate zone    :', fb['intermediate_zone_active'])

,clause,rule,observed,fired
0,1,zero_proposal_TINY >= 45/72,22/72,False
1,2,R_H3 < 0.15,0.1333333333,True


overall failure fired: True
intermediate zone    : False


## 9. Supply Criterion A

In [9]:
o = ADJ['observed']
print(f"PRE-H3 zero-proposal TINY : {o['pre_h3_zero_proposal_tiny']}/{72}")
print(f"H3     zero-proposal TINY : {o['zero_proposal_tiny']}/72")
print(f"TINY components recovered : {o['tiny_supply_recovered']}")
print()
print(f"Criterion A: zero_proposal_TINY <= 20/72  ->  observed {o['zero_proposal_tiny']}/72  ->  {ADJ['criterion_results']['criterion_A']['result']}")

PRE-H3 zero-proposal TINY : 57/72
H3     zero-proposal TINY : 22/72
TINY components recovered : 35

Criterion A: zero_proposal_TINY <= 20/72  ->  observed 22/72  ->  FAIL


## 10. Supply Criterion B

In [10]:
print(f"PRE-H3 zero-proposal GT   : {o['pre_h3_zero_proposal_gt']}/152")
print(f"H3     zero-proposal GT   : {o['zero_proposal_gt']}/152")
print(f"GT components recovered   : {o['gt_supply_recovered']}")
print()
print(f"Criterion B: zero_proposal_GT <= 35/152  ->  observed {o['zero_proposal_gt']}/152  ->  {ADJ['criterion_results']['criterion_B']['result']}")

PRE-H3 zero-proposal GT   : 79/152
H3     zero-proposal GT   : 27/152
GT components recovered   : 52

Criterion B: zero_proposal_GT <= 35/152  ->  observed 27/152  ->  PASS


## 11. Combined supply result

In [11]:
print('Combined primary supply result:', ADJ['criterion_results']['combined_primary_supply_result'])
print('Criterion A:', ADJ['criterion_results']['criterion_A']['result'])
print('Criterion B:', ADJ['criterion_results']['criterion_B']['result'])
print()
print('The primary supply criteria are NECESSARY but NOT sufficient alone;')
print('the downstream R_H3 is the adjudication gate.')

Combined primary supply result: FAIL
Criterion A: FAIL
Criterion B: PASS

The primary supply criteria are NECESSARY but NOT sufficient alone;
the downstream R_H3 is the adjudication gate.


## 12. Proposal-domain metric reconciliation

### Table E — Metric-domain reconciliation

In [12]:
ps = ADJ['proxy_and_sensitivity_status']
te = pd.DataFrame([
    {'quantity': 'H3_PROPOSAL_TINY_COVERAGE90', 'domain': 'PROPOSAL_LOCALIZER',
     'value': ps['proposal_domain_proxy']['value'], 'formal?': 'NO',
     'status': ps['proposal_domain_proxy']['status']},
    {'quantity': 'proxy R_H3 (proposal domain)', 'domain': 'PROPOSAL_LOCALIZER',
     'value': ps['proposal_domain_proxy']['proxy_R_H3'], 'formal?': 'NO',
     'status': 'LOCALIZER-ONLY PROXY - NOT FORMAL DOWNSTREAM R_H3'},
    {'quantity': 'strict cov90 TinyRecall', 'domain': 'SEGMENTATION (strict >=90%)',
     'value': ps['strict_cov90_sensitivity']['TinyRecall_H3_cov90'], 'formal?': 'NO',
     'status': ps['strict_cov90_sensitivity']['status']},
    {'quantity': 'formal TinyRecall_H3', 'domain': 'DETECTION (overlap >=1px)',
     'value': ADJ['observed']['formal_TinyRecall_H3'], 'formal?': 'YES',
     'status': 'FROZEN FORMAL DOMAIN'},
    {'quantity': 'formal R_H3', 'domain': 'DETECTION (overlap >=1px)',
     'value': ADJ['observed']['formal_R_H3'], 'formal?': 'YES',
     'status': 'FROZEN FORMAL GATE'},
])
display(te)

,quantity,domain,value,formal?,status
0,H3_PROPOSAL_TINY_COVERAGE90,PROPOSAL_LOCALIZER,0.666667,NO,INVALID_FOR_FORMAL_ADJUDICATION__PROPOSAL_DOMA...
1,proxy R_H3 (proposal domain),PROPOSAL_LOCALIZER,-0.600000,NO,LOCALIZER-ONLY PROXY - NOT FORMAL DOWNSTREAM R_H3
2,strict cov90 TinyRecall,SEGMENTATION (strict >=90%),0.500000,NO,DESCRIPTIVE_SENSITIVITY_ONLY__NOT_FORMAL_H3_GATE
3,formal TinyRecall_H3,DETECTION (overlap >=1px),0.819444,YES,FROZEN FORMAL DOMAIN
4,formal R_H3,DETECTION (overlap >=1px),0.133333,YES,FROZEN FORMAL GATE


## 13. Downstream metric-domain reconciliation

In [13]:
corr = load('reports/audits/h3_downstream_metric_domain_correction.json')
print('nature     :', corr['nature'])
print('as-run     :', corr['defect']['asrun_metric'], '->', corr['defect']['consequence_in_asrun'])
print('frozen     :', corr['defect']['frozen_metric'])
print('authority  :', corr['defect']['frozen_authority'])
print()
print('corrected formal:', json.dumps(corr['corrected_formal'], indent=2))

nature     : ADDITIVE_METRIC_LAYER_CORRECTION__NO_NEW_INFERENCE__NO_PIPELINE_CHANGE
as-run     : >=90% component-pixel coverage (strict) -> R_H3_asrun = -1.3999999999999995 (wrong domain)
frozen     : component overlap >= 1 px (detection)
authority  : reports/audits/phase14_clean_v3_global_reference_freeze.json

corrected formal: {
  "TinyRecall_H3": 0.8194444444444444,
  "Tg": 0.7916666666666666,
  "G": 0.20833333333333337,
  "R_H3": 0.13333333333333336,
  "interpretation_band": "insufficient"
}


## 14. Formal downstream TinyRecall_H3

In [14]:
print('formal TinyRecall_H3 :', ADJ['observed']['formal_TinyRecall_H3'])
print('fraction             :', ADJ['observed']['formal_TinyRecall_H3_fraction'])
print('metric domain        :', ADJ['observed']['metric_domain'])

formal TinyRecall_H3 : 0.8194444444444444
fraction             : 59/72
metric domain        : DETECTION (component overlap >= 1 pixel)


## 15. Formal R_H3

In [15]:
o = ADJ['observed']
print(f"Tg = {o['formal_Tg']:.16f}  ({o['formal_Tg_fraction']})")
print(f"G  = {o['formal_G']:.16f}  ({o['formal_G_fraction']})")
print(f"TinyRecall_H3 = {o['formal_TinyRecall_H3']:.16f}")
print()
print(f"R_H3 = (TinyRecall_H3 - Tg) / G = {o['formal_R_H3']:.16f}  ({o['formal_R_H3_fraction']})")
print(f"re-derived                     = {o['formal_R_H3_rederived']:.16f}")
print('interpretation band            :', o['R_H3_band'])

Tg = 0.7916666666666666  (57/72)
G  = 0.2083333333333334  (15/72)
TinyRecall_H3 = 0.8194444444444444

R_H3 = (TinyRecall_H3 - Tg) / G = 0.1333333333333334  (2/15)
re-derived                     = 0.1333333333333334
interpretation band            : insufficient


## 16. Clause-by-clause success table

In [16]:
sc = ADJ['frozen_criteria']; cr = ADJ['criterion_results']
display(pd.DataFrame([
    {'success clause': 'A', 'requirement': sc['criterion_A']['rule'], 'observed': cr['criterion_A']['observed'], 'met?': cr['criterion_A']['result']},
    {'success clause': 'B', 'requirement': sc['criterion_B']['rule'], 'observed': cr['criterion_B']['observed'], 'met?': cr['criterion_B']['result']},
    {'success clause': 'C', 'requirement': sc['criterion_C']['rule'], 'observed': f"{ADJ['observed']['formal_R_H3']:.6f}", 'met?': cr['criterion_C']['result']},
]))
print('formal success (ALL clauses):', cr['formal_success'])

,success clause,requirement,observed,met?
0,A,zero_proposal_TINY <= 20/72,22/72,FAIL
1,B,zero_proposal_GT <= 35/152,27/152,PASS
2,C,formal downstream R_H3 >= 0.30,0.133333,FAIL


formal success (ALL clauses): False


## 17. Clause-by-clause failure table

In [17]:
fb = ADJ['failure_boundary']
display(pd.DataFrame([
    {'failure clause': '1', 'rule': fb['clause_1']['rule'], 'observed': fb['clause_1']['observed'], 'fired?': fb['clause_1']['fired']},
    {'failure clause': '2', 'rule': fb['clause_2']['rule'], 'observed': f"{fb['clause_2']['observed']:.10f}", 'fired?': fb['clause_2']['fired']},
]))
print('failure boundary overall FIRED:', fb['fired'])
print('frozen failure label          :', fb['label'])

,failure clause,rule,observed,fired?
0,1,zero_proposal_TINY >= 45/72,22/72,False
1,2,R_H3 < 0.15,0.1333333333,True


failure boundary overall FIRED: True
frozen failure label          : H3_SUPPLY_REDESIGN_INSUFFICIENT


## 18. Supporting advanced metrics (DESCRIPTIVE ONLY)

In [18]:
pol = ADJ['post_hoc_supporting_metrics_policy']
for k, v in pol.items():
    print(f'{k:20s}: {v}')
print()
backfill = load('reports/audits/project_evaluation_metrics_backfill.json')
summ = pd.read_csv(TABLES / 'project_evaluation_segmentation_summary.csv')
h3 = summ[summ['arm'] == 'H3_DOWNSTREAM'].iloc[0]
cols = ['arm', 'pooled_dice', 'pooled_iou', 'pooled_precision', 'pooled_recall', 'pooled_specificity',
        'pooled_mcc', 'image_mean_hd95', 'image_mean_assd', 'image_mean_surface_dice',
        'brier', 'ece', 'roc_auc', 'average_precision']
display(summ[summ['arm'] == 'H3_DOWNSTREAM'][cols])
print('These metrics are SUPPORTING EVIDENCE ONLY and do NOT change the formal verdict.')

role                : SUPPORTING_EVIDENCE_ONLY
label               : DESCRIPTIVE_SUPPORTING_METRIC__NOT_H3_GATE
statistical_label   : DESCRIPTIVE_POST_HOC_STATISTICAL_SUPPORT
promoted_to_gate    : False
note                : Project-wide advanced metrics (confusion matrices, Dice/IoU, precision/recall/specificity/MCC, HD95/ASSD/surface Dice, calibration, bootstrap CIs, paired statistics) are used in Notebook 07 only as supporting evidence and do NOT become new prospective H3 criteria.



,arm,pooled_dice,pooled_iou,pooled_precision,pooled_recall,pooled_specificity,pooled_mcc,image_mean_hd95,image_mean_assd,image_mean_surface_dice,brier,ece,roc_auc,average_precision
3,H3_DOWNSTREAM,0.464786,0.30275,0.399027,0.556496,0.999358,0.470757,1566.038846,389.39584,0.150363,0.000828,0.000817,0.87959,0.409897


These metrics are SUPPORTING EVIDENCE ONLY and do NOT change the formal verdict.


## 19. Oracle / GLOBAL / PREDICTED / H3 comparison

### Table D — Main arm comparison

In [19]:
ct = load('reports/audits/h3_downstream_evaluation.json')['contrast_table']
ARM_KEYS = [('GLOBAL', 'GLOBAL'), ('PREDICTED', 'PREDICTED_REGIONAL'),
            ('H3', 'H3_DOWNSTREAM'), ('ORACLE_REGIONAL', 'ORACLE_REGIONAL')]
rows = []
for key, label in ARM_KEYS:
    d = ct[key]
    rows.append({'arm': label, 'mean_dice': d['mean_dice'], 'median_dice': d['median_dice'],
                 'tiny_recall90_det': d['tiny_recall90_det'],
                 'hard_tiny_recovered': f"{d['hard_tiny_recovered']}/15",
                 'fp_components_per_image': d['mean_fp_components_per_image']})
display(pd.DataFrame(rows).round(6))
print('ORACLE_REGIONAL is a DIAGNOSTIC UPPER BOUND (GT bboxes), not deployable performance.')

,arm,mean_dice,median_dice,tiny_recall90_det,hard_tiny_recovered,fp_components_per_image
0,GLOBAL,0.635917,0.745656,0.791667,0/15,0.485437
1,PREDICTED_REGIONAL,0.418531,0.370672,0.791667,0/15,7.776699
2,H3_DOWNSTREAM,0.388628,0.369360,0.819444,5/15,7.572816
3,ORACLE_REGIONAL,0.802416,0.834329,1.000000,15/15,0.631068


ORACLE_REGIONAL is a DIAGNOSTIC UPPER BOUND (GT bboxes), not deployable performance.


## 20. What improved

In [20]:
for s in ADJ['scientific_interpretation']['improved']:
    print(' -', s)

 - zero-proposal GT components fell 79 -> 27 (recovered 52)
 - zero-proposal TINY components fell 57 -> 22 (recovered 35)
 - raw proposal supply improved substantially on both the overall and TINY strata


## 21. What remained insufficient

In [21]:
for s in ADJ['scientific_interpretation']['insufficient']:
    print(' -', s)

 - Criterion A (zero-proposal TINY <= 20/72) FAILED at 22/72 (missed by 2)
 - formal downstream R_H3 = 2/15 = 0.1333333333 < 0.15 -> failure boundary FIRED


## 22. Formal H3 verdict

In [22]:
print('FROZEN SCIENTIFIC LABEL:', ADJ['final_frozen_H3_label'])
print()
print(ADJ['scientific_interpretation']['statement'])
print()
print('This verdict is the prospectively frozen rule applied to the authoritative observed values.')
print('It is NOT converted into PASS / PARTIAL SUCCESS / NEAR PASS / PROMOTE.')

FROZEN SCIENTIFIC LABEL: H3_SUPPLY_REDESIGN_INSUFFICIENT

The size-adaptive target-support intervention materially improved proposal supply, but the recovered supply did not translate into sufficient downstream Tiny-recall gain under the frozen Stage-4D pipeline. The intervention is therefore formally insufficient under the prospective H3 criteria.

This verdict is the prospectively frozen rule applied to the authoritative observed values.
It is NOT converted into PASS / PARTIAL SUCCESS / NEAR PASS / PROMOTE.


## 23. Regional-gate implication

In [23]:
rh = ADJ['regional_hypothesis_preserved']
for k, v in rh.items():
    print(f'{k:46s}: {v}')

h3_intervention_failed_its_frozen_criteria    : True
oracle_regional_processing_has_no_value       : False
oracle_evidence_retained                      : Phase-12/Phase-14 ORACLE_REGIONAL remains a valid diagnostic upper bound (mean Dice 0.8024158112932394; TINY detection 1.0)
predicted_localization_path                   : still fails to retain enough practical regional benefit (PREDICTED_REGIONAL mean Dice 0.4185309062818487; H3 downstream 0.38862753520437243)
regional_gate_state                           : REVISE_LOCALIZER (unresolved)


## 24. Phase-15 lock

In [24]:
print('Phase-15 status:', ADJ['governance']['phase_15_status'])
print()
print('The frozen criteria specify that when the H3 failure boundary fires:')
print('  - the target-support redesign is insufficient;')
print('  - Phase 15 remains NOT_STARTED_AND_NOT_AUTHORIZED;')
print('  - the H3 checkpoint must NOT be promoted as the solved deployable regional localizer;')
print('  - further localizer work requires fresh authorization.')

Phase-15 status: NOT_STARTED_AND_NOT_AUTHORIZED

The frozen criteria specify that when the H3 failure boundary fires:
  - the target-support redesign is insufficient;
  - Phase 15 remains NOT_STARTED_AND_NOT_AUTHORIZED;
  - the H3 checkpoint must NOT be promoted as the solved deployable regional localizer;
  - further localizer work requires fresh authorization.


## 25. TEST lock

In [25]:
g = ADJ['governance']
print('TEST reads          :', g['test_reads'])
print('external TEST reads :', g['external_test_reads'])
print('TEST status         : CLOSED')

TEST reads          : 0
external TEST reads : 0
TEST status         : CLOSED


## 26. Future options (NOT AUTHORIZED)

In [26]:
fo = ADJ['recommended_governance_next_state']['future_options']
for k, v in fo.items():
    print(f'{k:46s}: {v}')
print()
print('Option B and Option C are FUTURE OPTIONS ONLY. They are NOT implemented or trained here.')

option_B_positive_objective_reweighting       : FUTURE_ONLY__REQUIRES_FRESH_SUPERVISOR_AUTHORIZATION
option_C_recall_oriented_output_head          : FUTURE_ONLY__REQUIRES_FRESH_SUPERVISOR_AUTHORIZATION
authorized_now                                : False

Option B and Option C are FUTURE OPTIONS ONLY. They are NOT implemented or trained here.


## 27. Complete provenance table

### Table F — Provenance

In [27]:
rows = []
for name, rel in ADJ['evidence_chain_sha256'].items():
    rows.append({'artifact': name, 'path': rel if rel.endswith(('.json', '.csv', '.yaml', '.pt')) else rel,
                 'sha256': rel, 'role': 'EVIDENCE_CHAIN'})
prov = pd.DataFrame([{'artifact': k, 'sha256': v} for k, v in ADJ['evidence_chain_sha256'].items()])
prov['exists'] = prov['sha256'] != 'MISSING'
display(prov)
print()
print('adjudication artifact sha256:', sha('reports/audits/h3_formal_adjudication.json'))

,artifact,sha256,exists
0,design_protocol,51a1e0ae02f07506438da9e77ba0d8edd29c6382c2a8d6...,True
1,success_failure_criteria,9bfc361ddaee939d7cc0bd3f120e5ff4a504c403877a2c...,True
2,design_correction,82c2ca08c6a215605c2ce491cbea5d879bdfffa21a0294...,True
3,final_pretraining_binding_v2,ff10d13bd151d48f3d1c6ce2528a26b0a4fe16c8f97e37...,True
4,training_closure,d96d334d0e4d255443f13d2be8e0d8c2ad36b789c59e90...,True
5,checkpoint_selection_replay,4fb7312c70cbb747c58f11c1272ef7549a974607af86c5...,True
6,training_launch_binding,49046f31f6a766d485a761c8827bd3ca70a400f4779b17...,True
7,supply_axis_evaluation,00e6f41351d725817810eba13b738b2631c5b0cc023af1...,True
8,supply_metric_domain_reconciliation,457e87a7b51b378d3364d62884ab2fea98859684397762...,True
9,downstream_evaluation,9bfbf32f4ddaee7b3c34fadad22a6c09b1ce299349d5de...,True



adjudication artifact sha256: f07ff0c32499b3b05c42f176b21b2cd25329fc06fade7ab90f0f222586e71fb2


## 28. Final terminal state

In [28]:
print('adjudication state :', ADJ['status'])
print('scientific label   :', ADJ['scientific_label'])
print()
print('H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPORT_INSUFFICIENT__PHASE15_LOCKED')

adjudication state : H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPORT_INSUFFICIENT__PHASE15_LOCKED
scientific label   : H3_SUPPLY_REDESIGN_INSUFFICIENT

H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPORT_INSUFFICIENT__PHASE15_LOCKED


## Figures (saved data only, no model forward pass)

### Figure A — Supply improvement (zero-proposal GT / TINY, PRE-H3 vs H3)

In [29]:
o = ADJ['observed']
fig, ax = plt.subplots(figsize=(8, 4.6))
x = np.arange(2); w = 0.36
pre = [o['pre_h3_zero_proposal_gt'], o['pre_h3_zero_proposal_tiny']]
post = [o['zero_proposal_gt'], o['zero_proposal_tiny']]
ax.bar(x - w/2, pre, w, label='PRE-H3', color='#95a5a6')
ax.bar(x + w/2, post, w, label='H3', color='#2980b9')
for i, (a, b) in enumerate(zip(pre, post)):
    ax.text(i - w/2, a + 1, str(a), ha='center', fontsize=10)
    ax.text(i + w/2, b + 1, str(b), ha='center', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(['zero-proposal GT (of 152)', 'zero-proposal TINY (of 72)'])
ax.set_ylabel('count'); ax.set_title('H3 supply improvement (descriptive)'); ax.legend()
plt.tight_layout(); p = FIGDIR / 'figA_supply_improvement.png'
plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)

saved figA_supply_improvement.png


C:\Users\user\AppData\Local\Temp\ipykernel_32408\1362179296.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)


### Figure B — Formal Tiny Recall across arms

In [30]:
ct = load('reports/audits/h3_downstream_evaluation.json')['contrast_table']
ARM_KEYS = [('GLOBAL', 'GLOBAL'), ('PREDICTED', 'PREDICTED_REGIONAL'),
            ('H3', 'H3_DOWNSTREAM'), ('ORACLE_REGIONAL', 'ORACLE_REGIONAL')]
arms = [label for _, label in ARM_KEYS]
vals = [ct[key]['tiny_recall90_det'] for key, _ in ARM_KEYS]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
bars = ax.bar(arms, vals, color=['#7f8c8d', '#e67e22', '#2980b9', '#27ae60'])
ax.axhline(ADJ['observed']['formal_Tg'], color='black', ls='--', lw=1, label=f"Tg={ADJ['observed']['formal_Tg']:.4f}")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.4f}', ha='center', fontsize=9)
ax.set_ylim(0, 1.08); ax.set_ylabel('TINY recall (detection domain)')
ax.set_title('Formal Tiny Recall by arm (detection: overlap >= 1 px)'); ax.legend()
plt.tight_layout(); p = FIGDIR / 'figB_tiny_recall_by_arm.png'
plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)

saved figB_tiny_recall_by_arm.png


C:\Users\user\AppData\Local\Temp\ipykernel_32408\4266571511.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)


### Figure C — Formal R_H3 gauge

In [31]:
R = ADJ['observed']['formal_R_H3']
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.axvspan(-1.6, 0.15, color='#f5b7b1', alpha=0.6, label='insufficient (< 0.15)')
ax.axvspan(0.15, 0.30, color='#fdebd0', alpha=0.8, label='intermediate [0.15, 0.30)')
ax.axvspan(0.30, 1.2, color='#abebc6', alpha=0.6, label='success (>= 0.30)')
ax.axvline(R, color='red', lw=3)
ax.text(R, 0.55, f'R_H3 = {R:.4f}', color='red', ha='center', fontsize=11, fontweight='bold')
ax.axvline(0.15, color='black', ls='--', lw=1); ax.axvline(0.30, color='black', ls='--', lw=1)
ax.set_xlim(-1.6, 1.2); ax.set_yticks([]); ax.set_xlabel('formal R_H3')
ax.set_title('Formal R_H3 against the frozen thresholds (0.15 / 0.30)'); ax.legend(fontsize=8, loc='upper left')
plt.tight_layout(); p = FIGDIR / 'figC_rh3_gauge.png'
plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)

C:\Users\user\AppData\Local\Temp\ipykernel_32408\486546588.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)


saved figC_rh3_gauge.png


### Figure D — Success-gate panel

In [32]:
cr = ADJ['criterion_results']
gates = ['Criterion A\n(TINY <= 20/72)', 'Criterion B\n(GT <= 35/152)', 'Criterion C\n(R_H3 >= 0.30)']
res = [cr['criterion_A']['result'], cr['criterion_B']['result'], cr['criterion_C']['result']]
fig, ax = plt.subplots(figsize=(8, 4.2))
colors = ['#27ae60' if r == 'PASS' else '#c0392b' for r in res]
ax.bar(gates, [1, 1, 1], color=colors)
for i, r in enumerate(res):
    ax.text(i, 0.5, r, ha='center', va='center', color='white', fontsize=13, fontweight='bold')
ax.set_yticks([]); ax.set_title('Frozen success-gate panel (ALL clauses required)')
plt.tight_layout(); p = FIGDIR / 'figD_success_gates.png'
plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)

saved figD_success_gates.png


C:\Users\user\AppData\Local\Temp\ipykernel_32408\1971383311.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)


### Figure E — Failure-boundary panel

In [33]:
fb = ADJ['failure_boundary']
cl = ['Clause 1\nTINY >= 45/72', 'Clause 2\nR_H3 < 0.15']
fired = [fb['clause_1']['fired'], fb['clause_2']['fired']]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(cl, [1, 1], color=['#27ae60' if not f else '#c0392b' for f in fired])
for i, f in enumerate(fired):
    ax.text(i, 0.5, 'FIRED' if f else 'not fired', ha='center', va='center', color='white', fontsize=12, fontweight='bold')
ax.set_yticks([]); ax.set_title(f"Frozen failure boundary (overall FIRED = {fb['fired']})")
plt.tight_layout(); p = FIGDIR / 'figE_failure_boundary.png'
plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)

saved figE_failure_boundary.png


C:\Users\user\AppData\Local\Temp\ipykernel_32408\2073836628.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)


### Figure F — Downstream Dice comparison (descriptive only)

In [34]:
ct = load('reports/audits/h3_downstream_evaluation.json')['contrast_table']
ARM_KEYS = [('GLOBAL', 'GLOBAL'), ('PREDICTED', 'PREDICTED_REGIONAL'),
            ('H3', 'H3_DOWNSTREAM'), ('ORACLE_REGIONAL', 'ORACLE_REGIONAL')]
arms = [label for _, label in ARM_KEYS]
dice = [ct[key]['mean_dice'] for key, _ in ARM_KEYS]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
bars = ax.bar(arms, dice, color=['#7f8c8d', '#e67e22', '#2980b9', '#27ae60'])
for b, v in zip(bars, dice):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.4f}', ha='center', fontsize=9)
ax.set_ylabel('mean Dice'); ax.set_ylim(0, 0.95)
ax.set_title('Mean Dice by arm (DESCRIPTIVE_SUPPORTING_METRIC__NOT_H3_GATE)')
plt.tight_layout(); p = FIGDIR / 'figF_dice_by_arm.png'
plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)

C:\Users\user\AppData\Local\Temp\ipykernel_32408\605288470.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)


saved figF_dice_by_arm.png


### Figure G — Evidence-flow / decision diagram

In [35]:
fig, ax = plt.subplots(figsize=(11, 6.2))
ax.axis('off')
steps = [
    (0.5, 0.92, 'Root cause: F_MIXED_BOTTLENECK'),
    (0.5, 0.78, 'Axis priority: A_PRIORITY (R_A=1.0, R_C=0.0)'),
    (0.5, 0.64, 'H3 design: size-adaptive target support (A_ref=13.0)'),
    (0.5, 0.50, 'H3 training: 69 ep, best epoch 54, replay PASS'),
    (0.5, 0.36, 'Supply: zero-prop TINY 57->22 (A FAIL) / GT 79->27 (B PASS)'),
    (0.5, 0.22, 'Downstream: TinyRecall_H3 59/72 -> R_H3 = 2/15'),
]
for x, y, t in steps:
    ax.add_patch(plt.Rectangle((0.1, y - 0.05), 0.8, 0.1, facecolor='#d6eaf8', edgecolor='#2874a6'))
    ax.text(x, y, t, ha='center', va='center', fontsize=10)
    if y > 0.22:
        ax.annotate('', xy=(0.5, y - 0.06), xytext=(0.5, y - 0.10),
                    arrowprops=dict(arrowstyle='-|>', color='#2874a6'))
ax.add_patch(plt.Rectangle((0.1, 0.03), 0.8, 0.1, facecolor='#f5b7b1', edgecolor='#c0392b'))
ax.text(0.5, 0.08, 'VERDICT: H3_SUPPLY_REDESIGN_INSUFFICIENT (failure boundary FIRED)',
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_title('H3 evidence flow and decision path')
plt.tight_layout(); p = FIGDIR / 'figG_evidence_flow.png'
plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)

saved figG_evidence_flow.png


C:\Users\user\AppData\Local\Temp\ipykernel_32408\3361707100.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(str(p), dpi=150, bbox_inches='tight'); plt.show(); print('saved', p.name)


## Assertions

In [36]:
# ── Checkpoint binding ───────────────────────────────────────────────────────
assert ADJ['selected_checkpoint']['selected_epoch'] == 54
assert ADJ['selected_checkpoint']['sha256'] == 'd875cac7d09bbc3d32232191f0af2c138e5a74682fc63e4d6e01f30345eca75e'
assert ADJ['selected_checkpoint']['sha256_matches_expected'] is True
assert ADJ['checkpoint_selection_replay_verdict'] == 'H3_CHECKPOINT_SELECTION_REPLAY_PASS'

# ── Supply values from the frozen supply artifact ────────────────────────────
sup = load('reports/audits/h3_supply_axis_evaluation.json')
assert sup['primary_criteria']['criterion_a']['value'] == 22
assert sup['primary_criteria']['criterion_b']['value'] == 27
assert ADJ['observed']['zero_proposal_tiny'] == 22
assert ADJ['observed']['zero_proposal_gt'] == 27

# ── Criteria results ─────────────────────────────────────────────────────────
assert ADJ['criterion_results']['criterion_A']['result'] == 'FAIL'
assert ADJ['criterion_results']['criterion_B']['result'] == 'PASS'
assert ADJ['criterion_results']['combined_primary_supply_result'] == 'FAIL'

# ── Formal downstream metric domain ─────────────────────────────────────────
assert abs(ADJ['observed']['formal_TinyRecall_H3'] - 59/72) < 1e-12
assert abs(ADJ['observed']['formal_R_H3'] - 2/15) < 1e-12
assert ADJ['observed']['formal_R_H3'] < 0.15
assert 'DETECTION' in ADJ['observed']['metric_domain'].upper()

# ── Proxy and strict-cov90 are NOT formal ────────────────────────────────────
assert ADJ['proxy_and_sensitivity_status']['proposal_domain_proxy']['status'] == 'INVALID_FOR_FORMAL_ADJUDICATION__PROPOSAL_DOMAIN_ONLY'
assert ADJ['proxy_and_sensitivity_status']['strict_cov90_sensitivity']['status'] == 'DESCRIPTIVE_SENSITIVITY_ONLY__NOT_FORMAL_H3_GATE'

# ── Failure boundary and frozen label ────────────────────────────────────────
assert ADJ['failure_boundary']['clause_1']['fired'] is False
assert ADJ['failure_boundary']['clause_2']['fired'] is True
assert ADJ['failure_boundary']['fired'] is True
assert ADJ['final_frozen_H3_label'] == 'H3_SUPPLY_REDESIGN_INSUFFICIENT'
assert ADJ['scientific_label'] == 'H3_SUPPLY_REDESIGN_INSUFFICIENT'

# ── Governance ───────────────────────────────────────────────────────────────
g = ADJ['governance']
assert g['test_reads'] == 0
assert g['external_test_reads'] == 0
assert g['training_runs_this_task'] == 0
assert g['inference_runs_this_task'] == 0
assert g['checkpoint_changed'] is False
assert g['threshold_changed'] is False
assert g['frozen_gate_changed'] is False
assert g['phase_15_status'] == 'NOT_STARTED_AND_NOT_AUTHORIZED'
assert g['phase_15_status'] != 'STARTED'
assert g['option_b_started'] is False
assert g['option_c_started'] is False

# ── Verdict is not softened ──────────────────────────────────────────────────
assert ADJ['final_frozen_H3_label'] not in ('PASS', 'PARTIAL_SUCCESS', 'NEAR_PASS', 'PROMOTE')

print('✓ All H3 formal adjudication assertions passed.')
print('FROZEN LABEL: H3_SUPPLY_REDESIGN_INSUFFICIENT')
print('State: H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPORT_INSUFFICIENT__PHASE15_LOCKED')

✓ All H3 formal adjudication assertions passed.
FROZEN LABEL: H3_SUPPLY_REDESIGN_INSUFFICIENT
State: H3_FORMAL_ADJUDICATION_COMPLETE__TARGET_SUPPORT_INSUFFICIENT__PHASE15_LOCKED
